# Step 3: Feature Selection

1. use selected feature, fit xgboost

2. Optuna to tune

3. cross validation

4. final model; score test set



In [1]:
import os
import numpy as np
import pandas as pd

import optuna

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import f1_score, make_scorer

import xgboost as xgb
from sklearn.multioutput import MultiOutputClassifier


/Applications/anaconda3/envs/wids-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir = os.getcwd()
home_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
data_dir = os.path.join(home_dir, 'data/')

In [3]:
train_cat = pd.read_csv(os.path.join(data_dir, 'intermediate/encoded_train_cat.csv'))
train_quant = pd.read_csv(os.path.join(data_dir, 'intermediate/train_quant_filled.csv'))
train_fcm_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/train_pca_100.csv'))
train_fcm_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/train_pca_500.csv'))

test_cat = pd.read_csv(os.path.join(data_dir, 'intermediate/encoded_test_cat.csv'))
test_quant = pd.read_csv(os.path.join(data_dir, 'intermediate/test_quant_filled.csv'))
test_fcm_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/test_pca_100.csv'))
test_fcm_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/test_pca_500.csv'))

# selected features
features_100 = pd.read_csv(os.path.join(data_dir, 'intermediate/common_features_selected.csv'), header=None)
features_500 = pd.read_csv(os.path.join(data_dir, 'intermediate/common_features_selected_500.csv'), header=None)

train_label = pd.read_excel(os.path.join(data_dir, 'TRAIN_NEW/TRAINING_SOLUTIONS.xlsx'))

In [5]:
# Join data
X = train_cat.merge(train_quant, on='participant_id', how='left').merge(train_fcm_100, on='participant_id', how='left')
X = X.drop(columns=['participant_id'], axis=1)
y = train_label.copy()
y = y.drop(columns=['participant_id'], axis=1)

In [6]:
features_100_list = features_100[0].tolist()
features_100_list = [x.removesuffix('_scaled') for x in features_100_list]

In [7]:
# only keep the selected features
X = X[features_100_list]

In [ ]:
# Split training and test data
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

In [12]:
# Custom scoring function for weighted F1 score
def weighted_f1(y_true, y_pred):
    # Ensure that y_true and y_pred are numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    f1_adhd = f1_score(y_true[:, 0], y_pred[:, 0])
    f1_sex = f1_score(y_true[:, 1], y_pred[:, 1])
    return (2/3) * f1_adhd + (1/3) * f1_sex

In [13]:
# Objective function for Optuna optimization
def objective(trial):
    # Define the hyperparameters to tune
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'n_jobs': -1
    }

    # Define the base model
    base_model = xgb.XGBClassifier(**params)

    # Wrap it with MultiOutputClassifier
    multi_model = MultiOutputClassifier(base_model)

    # Perform cross-validation (here we're using 5-fold cross-validation)
    scores = cross_val_score(multi_model, X, y, cv=5, scoring=make_scorer(weighted_f1))

    # Return the mean of the cross-validation scores
    return np.mean(scores)


In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best params:", study.best_trial.params)

[I 2025-04-26 13:02:58,760] A new study created in memory with name: no-name-ae58a67c-db62-4edd-97bf-cbc56c689ba5
[I 2025-04-26 13:03:00,823] Trial 0 finished with value: 0.49527403694373173 and parameters: {'max_depth': 8, 'learning_rate': 0.26719183960842147, 'n_estimators': 216, 'subsample': 0.5070791894063129, 'colsample_bytree': 0.9367759842867175, 'gamma': 4.080955571440241}. Best is trial 0 with value: 0.49527403694373173.
